# 02 · Silver · `geracao_distribuida`
### V-Energy Grid 360 — Arquitetura Medalhão

Lê `v_energy.bronze.geracao_distribuida` (`geracao_distribuida.csv`, tudo como texto) e grava `v_energy.silver.geracao_distribuida`. **Grão: 1 linha = 1 UC com geração distribuída** (cada UC tem um único registro).

**Fonte das regras:** `exploracao_geracao_distribuida.ipynb` e a seção `geracao_distribuida` do `docs/data_quality_log.md` (G1 a G5).

| # | Tratamento | Linhas afetadas | Regra | Log |
|---|---|---|---|---|
| 1 | `potencia_kwp` com vírgula decimal | 316 | vírgula vira ponto | G1 |
| 2 | `potencia_kwp` registrada em watts | 35 | dividir por 1.000 + flag `potencia_convertida_de_w` | G2 |
| 3 | `data_conexao` antes da ligação da UC | 5 | manter a data + flag `conexao_antes_ligacao` | G3 |
| 4 | Tipos | todas | data → `date`; potência e energia → `decimal` | G4 |
| 5 | Energia injetada atípica (106 delas exatamente potência × 110) | 109 | manter o valor + flag `energia_injetada_atipica` | G5 |

**Nenhuma linha é removida:** a tabela não tem cópias, nulos, órfãs nem UCs de teste (exploração, seções 3 a 8).

**Por que esta tabela importa:** as 3.510 UCs com GD explicam toda a queda de volume do diagnóstico (exploração, seção 12). Erros aqui afetam diretamente a resposta ao diretor.

**Resultado esperado:** 3.510 linhas, `id_uc` único, potência entre 1 e 75 kWp, 109 UCs com energia injetada atípica.

**Pré-requisito:** `silver.unidades_consumidoras` gravada (integridade e flag G3).

## 1. Parâmetros do ambiente

In [0]:
catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

from pyspark.sql import functions as F, Row
from datetime import datetime

tabela_bronze = f"{bronze_schema}.geracao_distribuida"
tabela_silver = f"{silver_schema}.geracao_distribuida"

def n_br(n):
    return f"{n:,}".replace(",", ".")

print(f"origem:  {tabela_bronze}")
print(f"destino: {tabela_silver}")

## 2. Funções de checagem de qualidade

Mesmas funções das outras Silvers, gravando em `silver.dq_log`. O log é gravado **sempre**; a Silver só é gravada se **todas** as checagens passarem.

In [0]:
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

def dq_check_contagem(table_name: str, check_name: str, df, esperado: int):
    """Checagem de contagem total: passa só se o número de linhas for exatamente o esperado."""
    total = df.count()
    diferenca = abs(total - esperado)
    passed = diferenca == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=diferenca, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {n_br(total)} linhas (esperado {n_br(esperado)})")

## 3. Leitura da Bronze

Leitura sem transformação. Conferência: **3.510** linhas, todas as colunas `string`.

In [0]:
bronze = spark.table(tabela_bronze)

n_bronze = bronze.count()
print(f"Linhas na Bronze: {n_br(n_bronze)}")  # esperado: 3.510

bronze.printSchema()
display(bronze.limit(5))

## 4. Potência: vírgula decimal e valores em watts (log G1 e G2)

**G1 · vírgula decimal:** 316 potências vêm com vírgula (`5,87`). A vírgula vira ponto antes da conversão; sem isso, o `cast` anularia esses valores.

**G2 · valores em watts:** 35 potências têm 4 dígitos antes do separador (de 1.000 a 8.995). Uma residência com 9 mil kWp não existe (o limite da minigeração é 5.000 kW), e a energia injetada confirma que o valor está em **W**:

| Grupo | UCs | Energia injetada por kWp (mediana, kWh/mês) |
|---|---|---|
| Potências normais | 3.475 | 33,8 |
| Potências suspeitas, como vieram | 35 | 0,03 |
| Potências suspeitas, divididas por 1.000 | 35 | **33,3** |

Depois da divisão, a relação energia/potência fica igual à das outras UCs, e as potências ficam entre 1,5 e 9 kWp, como o resto da base.

**Decisão:** dividir por 1.000 as potências **a partir de 1.000** e marcar `potencia_convertida_de_w = true`. O valor original fica em `potencia_kwp_original`.

**Por que o corte em 1.000:** não há nenhuma potência entre 75 e 1.000 (exploração, seção 9). O corte separa os dois grupos sem ambiguidade.

**Conferência:** 316 vírgulas convertidas; 35 convertidas de W; 0 nulas; máximo de até 75 kWp.

In [0]:
# Tratamentos 1 e 2 · potência: vírgula decimal e conversão de W para kW
pot_num = F.expr("try_cast(regexp_replace(trim(potencia_kwp), ',', '.') as double)")

df_pot = (
    bronze
    .withColumn("potencia_kwp_original", F.col("potencia_kwp"))
    .withColumn("pot", pot_num)
    .withColumn("potencia_convertida_de_w", F.col("pot") >= 1000)
    .withColumn("potencia_kwp",
                F.when(F.col("potencia_convertida_de_w"), F.col("pot") / 1000).otherwise(F.col("pot"))
                 .cast("decimal(10,3)"))
    .drop("pot")
)

# esperado: 316 com vírgula | 35 convertidas de W | 0 nulas | máximo <= 75
print("com vírgula decimal:   ", n_br(df_pot.filter(F.col("potencia_kwp_original").contains(",")).count()))
print("convertidas de W:      ", n_br(df_pot.filter("potencia_convertida_de_w").count()))
print("potência nula:         ", n_br(df_pot.filter("potencia_kwp IS NULL").count()))
display(df_pot.agg(F.min("potencia_kwp"), F.expr("percentile_approx(potencia_kwp, 0.5)").alias("mediana"), F.max("potencia_kwp")))
display(df_pot.filter("potencia_convertida_de_w").select("id_uc", "potencia_kwp_original", "potencia_kwp").limit(10))

## 5. Tipos: data de conexão, energia injetada e modalidade (log G4)

- `data_conexao`: todas em ISO (`yyyy-MM-dd`) na exploração. Mesmo assim, a conversão usa `try_to_timestamp`, e a checagem de não nulo pega qualquer formato novo.
- `energia_injetada_kwh_media_mensal`: sempre com ponto decimal; vira `decimal(12,2)`.
- `modalidade`: 3 valores, sem variações; `upper(trim)` só garante o padrão.

**Conferência:** 0 datas e 0 energias nulas depois da conversão.

In [0]:
# Tratamento 4 · tipos
df_tipos = (
    df_pot
    .withColumn("data_conexao_original", F.col("data_conexao"))
    .withColumn("data_conexao", F.expr("to_date(try_to_timestamp(trim(data_conexao), 'yyyy-MM-dd'))"))
    .withColumn("energia_injetada_kwh_media_mensal",
                F.expr("try_cast(regexp_replace(trim(energia_injetada_kwh_media_mensal), ',', '.') as decimal(12,2))"))
    .withColumn("modalidade", F.upper(F.trim("modalidade")))
    .withColumn("id_uc", F.upper(F.trim("id_uc")))
)

# esperado: 0 | 0
print("data_conexao nula:     ", n_br(df_tipos.filter("data_conexao IS NULL").count()))
print("energia injetada nula: ", n_br(df_tipos.filter("energia_injetada_kwh_media_mensal IS NULL").count()))
display(df_tipos.groupBy("modalidade").count().orderBy(F.desc("count")))

## 6. Conexão antes da ligação da UC (log G3)

**O problema:** em 5 UCs, a GD aparece conectada antes de a própria UC ser ligada (de 55 a 473 dias antes). Fisicamente, não é possível.

**Decisão:** **manter as duas datas** e marcar `conexao_antes_ligacao = true`.

**Por que não corrigir:** não há gabarito para saber qual das duas datas está errada (a de conexão ou a de ligação). Trocar uma pela outra seria inventar um dado. O efeito é pequeno: 5 de 3.510 UCs. A flag permite excluí-las de qualquer análise que dependa da data de conexão.

**Conferência:** 5 UCs marcadas.

In [0]:
# Tratamento 3 · flag para conexão antes da ligação
ucs = spark.table(f"{silver_schema}.unidades_consumidoras").select("id_uc", "data_ligacao")

df_flag = (df_tipos.join(ucs, "id_uc", "left")
           .withColumn("conexao_antes_ligacao", F.coalesce(F.col("data_conexao") < F.col("data_ligacao"), F.lit(False))))

# esperado: 5
print("conexão antes da ligação:", n_br(df_flag.filter("conexao_antes_ligacao").count()))
display(df_flag.filter("conexao_antes_ligacao")
        .select("id_uc", "data_conexao", "data_ligacao", F.datediff("data_ligacao", "data_conexao").alias("dias_antes")))

## 7. Energia injetada atípica (log G5)

**O problema:** na grande maioria das UCs, a energia injetada fica entre 11,6 e ~50 kWh por kWp ao mês (mediana 33,8; percentis 1% e 95%: 22,3 e 44,3). Em **109 UCs**, passa de 60:
- **106 têm exatamente 110,0 kWh por kWp**, ou seja, a energia foi calculada como `potência × 110`, e não medida. 110 kWh/kWp por mês é a **geração** típica de um painel em Recife, e não a sobra injetada na rede.
- As outras 3 (114, 168 e 328) são UCs com potência convertida de watts (G2).
- **101 das 109 foram conectadas em 2026**, ainda sem histórico para calcular uma média mensal real. As **5 UCs com conexão antes da ligação (G3) estão todas neste grupo.**

**Hipóteses:** (a) estimativa teórica do sistema para conexões recentes, até haver medição; (b) registros suspeitos, de interesse para a fila de perdas. Os dados não decidem entre as duas.

**Decisão:** **manter o valor** e marcar `energia_injetada_atipica = true`. Não sabemos qual seria a energia injetada real, e trocar por uma estimativa seria inventar um dado. A flag avisa que o valor não deve ser usado como medida.

**Impacto:** o diagnóstico da receita não usa esta coluna (a queda de volume é medida nas leituras). Na Gold, a `energia_injetada_media_kwh` da `dim_uc` precisa vir com a flag.

**Conferência:** 109 UCs marcadas.

In [0]:
# Tratamento 5 · flag para energia injetada fora da faixa típica
df_energia = df_flag.withColumn(
    "energia_injetada_atipica",
    F.coalesce((F.col("energia_injetada_kwh_media_mensal") / F.col("potencia_kwp")) > 60, F.lit(False)))

# esperado: 109 marcadas | 106 com exatamente 110 kWh/kWp | 101 conectadas em 2026
atip = df_energia.filter("energia_injetada_atipica").withColumn(
    "razao", F.round(F.col("energia_injetada_kwh_media_mensal") / F.col("potencia_kwp"), 1))
print("energia injetada atípica:", n_br(atip.count()))
display(atip.groupBy("razao").count().orderBy(F.desc("count")))
display(atip.groupBy(F.year("data_conexao").alias("ano_conexao"), "conexao_antes_ligacao", "potencia_convertida_de_w").count())

## 8. Seleção final das colunas

Os valores originais de potência e data (`*_original`) e a `data_ligacao` (que pertence à tabela de UCs) ficam de fora. `source_file` é mantida para auditoria.

In [0]:
colunas_finais = [
    "id_uc", "potencia_kwp", "potencia_convertida_de_w", "data_conexao", "conexao_antes_ligacao",
    "modalidade", "energia_injetada_kwh_media_mensal", "energia_injetada_atipica", "source_file",
]
df_silver = df_energia.select(*colunas_finais)
df_silver.printSchema()

## 9. Checagens de qualidade

| Checagem | Protege |
|---|---|
| Total = 3.510 e `id_uc` único | grão de 1 linha por UC |
| Colunas não nulas | conversões (G1, G2, G4) |
| `id_uc` no padrão e existente na Silver de UCs | integridade com o cadastro |
| Potência entre 1 e 75 kWp | conversão de W (G2): nenhum valor em watts sobrou |
| Energia injetada positiva | tipo numérico (G4) |
| Energia por kWp entre 10 e 60 kWh/mês, **ou** marcada como atípica | coerência entre potência e energia (G2); qualquer valor fora da faixa sem flag é anomalia nova (G5) |
| `modalidade` entre os 3 valores | domínio |
| Conexão entre 2012-01-01 e 2026-06-30 | datas impossíveis |

A contagem de 3.510 vale para esta extração; com dados novos, o valor esperado muda.

In [0]:
t = "silver.geracao_distribuida"
dq_results.clear()

def ok(cond):
    return F.coalesce(cond, F.lit(False))

dq_check_contagem(t, "total de linhas = 3.510", df_silver, 3_510)
dq_check_unique(t, "id_uc único (1 registro por UC)", df_silver, ["id_uc"])

for coluna in ["id_uc", "potencia_kwp", "potencia_convertida_de_w", "data_conexao",
               "conexao_antes_ligacao", "modalidade", "energia_injetada_kwh_media_mensal",
               "energia_injetada_atipica"]:
    dq_check(t, f"{coluna} não nulo", df_silver, F.col(coluna).isNotNull())

dq_check(t, "id_uc no padrão UC + 6 dígitos", df_silver, ok(F.col("id_uc").rlike("^UC[0-9]{6}$")))
ids_uc = spark.table(f"{silver_schema}.unidades_consumidoras").select("id_uc").withColumn("existe", F.lit(True))
dq_check(t, "id_uc existe na Silver de UCs", df_silver.join(ids_uc, "id_uc", "left"), ok(F.col("existe")))

dq_check(t, "potência entre 1 e 75 kWp", df_silver, ok(F.col("potencia_kwp").between(1, 75)))
dq_check(t, "energia injetada positiva", df_silver, ok(F.col("energia_injetada_kwh_media_mensal") > 0))
dq_check(t, "energia por kWp entre 10 e 60 kWh/mês, ou marcada como atípica", df_silver,
         ok((F.col("energia_injetada_kwh_media_mensal") / F.col("potencia_kwp")).between(10, 60))
         | F.col("energia_injetada_atipica"))
dq_check(t, "energia atípica só acima da faixa (nenhuma abaixo de 10 kWh/kWp)", df_silver,
         ok((F.col("energia_injetada_kwh_media_mensal") / F.col("potencia_kwp")) >= 10))
dq_check(t, "modalidade entre os 3 valores", df_silver,
         ok(F.col("modalidade").isin("GERACAO NA PROPRIA UC", "AUTOCONSUMO REMOTO", "GERACAO COMPARTILHADA")))
dq_check(t, "data_conexao entre 2012-01-01 e 2026-06-30", df_silver,
         ok(F.col("data_conexao").between(F.lit("2012-01-01").cast("date"), F.lit("2026-06-30").cast("date"))))

## 10. Registro das checagens em `silver.dq_log`

Gravado **sempre**, mesmo com falha (modo `append`).

In [0]:
df_dq_log = spark.createDataFrame(dq_results)
df_dq_log.write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable(f"{silver_schema}.dq_log")

display(
    spark.table(f"{silver_schema}.dq_log")
    .filter(F.col("table_name") == t)
    .orderBy(F.col("checked_at").desc())
    .limit(len(dq_results))
)

## 11. Gravação da Silver, só se todas as checagens passarem

In [0]:
falhas = [r for r in dq_results if not r.passed]

if falhas:
    print(f"{len(falhas)} checagem(ns) falharam. {tabela_silver} NÃO foi gravada:")
    for r in falhas:
        print(f"  - {r.check_name}: {n_br(r.failed_rows)} falha(s) em {n_br(r.total_rows)} linhas")
    raise Exception(f"{tabela_silver} não gravada: {len(falhas)} checagem(ns) de qualidade falharam (veja silver.dq_log)")

(
    df_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_silver)
)
print(f"Todas as {len(dq_results)} checagens passaram. {tabela_silver} gravada: "
      f"{n_br(spark.table(tabela_silver).count())} linhas")

## 12. Bronze × Silver, com as mesmas UCs

Para cada tratamento, as mesmas `id_uc` nas duas tabelas: o texto original da Bronze ao lado do valor tratado.

In [0]:
silver_t = spark.table(tabela_silver)

def comparar(titulo, ids_df, n=3):
    ids = [r["id_uc"] for r in ids_df.select("id_uc").limit(n).collect()]
    print(f"=== {titulo} | ids: {ids}")
    print("Bronze (texto original):")
    display(bronze.filter(F.col("id_uc").isin(ids)).drop("source_file", "ingestion_timestamp").orderBy("id_uc"))
    print("Silver (tratado):")
    display(silver_t.filter(F.col("id_uc").isin(ids)).drop("source_file").orderBy("id_uc"))

comparar("Potência com vírgula decimal", bronze.filter(F.col("potencia_kwp").contains(",")))
comparar("Potência em watts", silver_t.filter("potencia_convertida_de_w"))
comparar("Conexão antes da ligação", silver_t.filter("conexao_antes_ligacao"))
comparar("Energia injetada atípica", silver_t.filter("energia_injetada_atipica"))